In [1]:
from pathlib import Path
from datetime import datetime

import polars as pl
from pokerkit import HandHistory
from pathlib import Path

In [3]:
# Start small while testing.
MAX_FILES = 10

# Later you can increase this to something like 100 or 500.
FILES_PER_CHUNK = 5

In [4]:
def to_float(value):
    """Safely convert Decimal/int/float/None to float."""
    if value is None:
        return None
    return float(value)


def make_datetime(hand):
    """Combine PHH date/time fields into one Python datetime."""
    if hand.year is None or hand.month is None or hand.day is None:
        return None

    hour = hand.time.hour if hand.time is not None else 0
    minute = hand.time.minute if hand.time is not None else 0
    second = hand.time.second if hand.time is not None else 0

    return datetime(
        hand.year,
        hand.month,
        hand.day,
        hour,
        minute,
        second,
    )


def get_player_value(values, i):
    """
    Safely get values[i].
    Useful because some optional PHH arrays may be None.
    """
    if values is None:
        return None

    if i >= len(values):
        return None

    return values[i]

In [6]:
# -------------------------
# Row extraction
# -------------------------

def extract_hand_rows(hand, source_file):
    """
    Convert one PokerKit HandHistory into rows for:
        1. hands
        2. player_hands
        3. actions
    """

    num_players = len(hand.starting_stacks)

    # -------------------------
    # HAND TABLE
    # -------------------------

    hand_row = {
        "hand_id": str(hand.hand),
        "variant": hand.variant,
        "venue": hand.venue,
        "table": hand.table,
        "datetime": make_datetime(hand),
        "currency": hand.currency,
        "min_bet": to_float(hand.min_bet),
        "small_bet": to_float(hand.small_bet),
        "big_bet": to_float(hand.big_bet),
        "bring_in": to_float(hand.bring_in),
        "players_dealt": num_players,
        "seat_count": hand.seat_count,
        "source_file": str(source_file),
    }

    # -------------------------
    # PLAYER-HAND TABLE
    # -------------------------

    player_rows = []

    for i in range(num_players):

        player_id = get_player_value(hand.players, i)
        seat = get_player_value(hand.seats, i)
        starting_stack = get_player_value(hand.starting_stacks, i)
        ante = get_player_value(hand.antes, i)
        blind = get_player_value(hand.blinds_or_straddles, i)
        winnings = get_player_value(hand.winnings, i)

        player_rows.append({
            "hand_id": str(hand.hand),
            "player_id": player_id,
            "player_number": i + 1,  # PHH p1, p2, p3...
            "seat": seat,
            "starting_stack": to_float(starting_stack),
            "ante": to_float(ante),
            "blind_or_straddle": to_float(blind),
            "winnings": to_float(winnings),
        })

    # -------------------------
    # ACTION TABLE
    # -------------------------

    action_rows = []

    for action_number, action in enumerate(hand.actions):

        action_rows.append({
            "hand_id": str(hand.hand),
            "action_number": action_number,
            "raw_action": action,
        })

    return hand_row, player_rows, action_rows



In [7]:
# -------------------------
# Process one group of files
# -------------------------

def process_chunk(files, chunk_number):

    hands_rows = []
    player_rows = []
    action_rows = []

    total_hands = 0

    for file_number, file_path in enumerate(files, start=1):

        print(
            f"Chunk {chunk_number} | "
            f"File {file_number}/{len(files)} | "
            f"{file_path.name}"
        )

        try:
            with open(file_path, "rb") as f:

                # IMPORTANT:
                # This stays a generator.
                # We never convert the entire file to a list.
                for hand in HandHistory.load_all(f):

                    hand_row, players, actions = extract_hand_rows(
                        hand,
                        file_path,
                    )

                    hands_rows.append(hand_row)
                    player_rows.extend(players)
                    action_rows.extend(actions)

                    total_hands += 1

        except Exception as e:
            print(f"ERROR: {file_path}")
            print(e)

    print(f"Writing chunk {chunk_number}")
    print(f"Hands: {len(hands_rows):,}")
    print(f"Player rows: {len(player_rows):,}")
    print(f"Action rows: {len(action_rows):,}")

    if hands_rows:
        pl.DataFrame(hands_rows).write_parquet(
            HANDS_DIR / f"hands_{chunk_number:04d}.parquet"
        )

    if player_rows:
        pl.DataFrame(player_rows).write_parquet(
            PLAYER_HANDS_DIR
            / f"player_hands_{chunk_number:04d}.parquet"
        )

    if action_rows:
        pl.DataFrame(action_rows).write_parquet(
            ACTIONS_DIR / f"actions_{chunk_number:04d}.parquet"
        )

    return total_hands


In [8]:
# -------------------------
# Main
# -------------------------

files = sorted(ROOT.rglob("*.phhs"))

if MAX_FILES is not None:
    files = files[:MAX_FILES]

print(f"Found {len(files):,} files to process.")

grand_total_hands = 0

for chunk_number, start in enumerate(
    range(0, len(files), FILES_PER_CHUNK)
):

    chunk = files[start:start + FILES_PER_CHUNK]

    hands_processed = process_chunk(
        chunk,
        chunk_number,
    )

    grand_total_hands += hands_processed

print()
print("DONE")
print(f"Total hands processed: {grand_total_hands:,}")

Found 10 files to process.
Chunk 0 | File 1/5 | abs NLH handhq_1-OBFUSCATED.phhs


C:\Users\thukr\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\pokerkit\notation.py:434: UserWarning: The field 'time_zone_abbreviation' is an unexpected field and should probably be prefixed with an underscore character '_'.
  warn(


Chunk 0 | File 2/5 | abs NLH handhq_10-OBFUSCATED.phhs
Chunk 0 | File 3/5 | abs NLH handhq_11-OBFUSCATED.phhs
Chunk 0 | File 4/5 | abs NLH handhq_12-OBFUSCATED.phhs
Chunk 0 | File 5/5 | abs NLH handhq_13-OBFUSCATED.phhs
Writing chunk 0
Hands: 4,998
Player rows: 21,637
Action rows: 60,770
Chunk 1 | File 1/5 | abs NLH handhq_14-OBFUSCATED.phhs
Chunk 1 | File 2/5 | abs NLH handhq_15-OBFUSCATED.phhs
Chunk 1 | File 3/5 | abs NLH handhq_16-OBFUSCATED.phhs
Chunk 1 | File 4/5 | abs NLH handhq_17-OBFUSCATED.phhs
Chunk 1 | File 5/5 | abs NLH handhq_18-OBFUSCATED.phhs
Writing chunk 1
Hands: 5,000
Player rows: 22,724
Action rows: 63,470

DONE
Total hands processed: 9,998


In [9]:
hands = pl.scan_parquet("data/hands/*.parquet")
players = pl.scan_parquet("data/player_hands/*.parquet")
actions = pl.scan_parquet("data/actions/*.parquet")

In [10]:
print(hands.head(10).collect())

shape: (10, 13)
┌────────────┬─────────┬────────────┬───────────┬───┬──────────┬───────────┬───────────┬───────────┐
│ hand_id    ┆ variant ┆ venue      ┆ table     ┆ … ┆ bring_in ┆ players_d ┆ seat_coun ┆ source_fi │
│ ---        ┆ ---     ┆ ---        ┆ ---       ┆   ┆ ---      ┆ ealt      ┆ t         ┆ le        │
│ str        ┆ str     ┆ str        ┆ str       ┆   ┆ null     ┆ ---       ┆ ---       ┆ ---       │
│            ┆         ┆            ┆           ┆   ┆          ┆ i64       ┆ null      ┆ str       │
╞════════════╪═════════╪════════════╪═══════════╪═══╪══════════╪═══════════╪═══════════╪═══════════╡
│ 3017243630 ┆ NT      ┆ Absolute   ┆ CLEO AVE  ┆ … ┆ null     ┆ 6         ┆ null      ┆ handhq\AB │
│            ┆         ┆ Poker      ┆           ┆   ┆          ┆           ┆           ┆ S-2009-07 │
│            ┆         ┆            ┆           ┆   ┆          ┆           ┆           ┆ -01_2009- │
│            ┆         ┆            ┆           ┆   ┆          ┆           

In [11]:
print(
    hands
    .select(pl.len().alias("hands"))
    .collect()
)

shape: (1, 1)
┌───────┐
│ hands │
│ ---   │
│ u32   │
╞═══════╡
│ 9998  │
└───────┘


shape: (0, 8)
┌─────────┬───────────┬───────────────┬──────┬────────────────┬──────┬──────────────────┬──────────┐
│ hand_id ┆ player_id ┆ player_number ┆ seat ┆ starting_stack ┆ ante ┆ blind_or_straddl ┆ winnings │
│ ---     ┆ ---       ┆ ---           ┆ ---  ┆ ---            ┆ ---  ┆ e                ┆ ---      │
│ str     ┆ str       ┆ i64           ┆ i64  ┆ f64            ┆ f64  ┆ ---              ┆ f64      │
│         ┆           ┆               ┆      ┆                ┆      ┆ f64              ┆          │
╞═════════╪═══════════╪═══════════════╪══════╪════════════════╪══════╪══════════════════╪══════════╡
└─────────┴───────────┴───────────────┴──────┴────────────────┴──────┴──────────────────┴──────────┘


In [14]:

from collections import Counter
from pokerkit import HandHistory

ROOT = Path("handhq")

files = sorted(ROOT.rglob("*.phhs"))[:10]

stats = Counter()
examples = []

for file_path in files:

    with open(file_path, "rb") as f:

        for hand in HandHistory.load_all(f):

            n_players = len(hand.starting_stacks)

            # Entire winnings field missing
            if hand.winnings is None:
                stats["hands_winnings_none"] += 1
                stats["affected_player_rows"] += n_players

                if len(examples) < 5:
                    examples.append({
                        "hand": hand.hand,
                        "players": n_players,
                        "winnings": hand.winnings,
                        "actions": hand.actions,
                    })

                continue

            stats["hands_winnings_present"] += 1

            # Check array length
            if len(hand.winnings) != n_players:
                stats["winnings_length_mismatch"] += 1

            # Check individual None entries
            none_count = sum(
                value is None
                for value in hand.winnings
            )

            if none_count:
                stats["hands_with_partial_none"] += 1
                stats["individual_none_values"] += none_count

                if len(examples) < 5:
                    examples.append({
                        "hand": hand.hand,
                        "players": n_players,
                        "winnings": hand.winnings,
                        "actions": hand.actions,
                    })


print("=== WINNINGS DIAGNOSTICS ===")

for key, value in stats.items():
    print(f"{key}: {value:,}")


print("\n=== EXAMPLE PROBLEM HANDS ===")

for example in examples:

    print("\nHand:", example["hand"])
    print("Players:", example["players"])
    print("Winnings:", example["winnings"])

    print("Actions:")

    for action in example["actions"]:
        print("   ", action)

=== WINNINGS DIAGNOSTICS ===
hands_winnings_present: 8,702
hands_winnings_none: 1,296
affected_player_rows: 5,975

=== EXAMPLE PROBLEM HANDS ===

Hand: 3017254774
Players: 6
Winnings: None
Actions:
    d dh p1 ????
    d dh p2 ????
    d dh p3 ????
    d dh p4 ????
    d dh p5 ????
    d dh p6 ????
    p3 f
    p4 cbr 40
    p5 cc
    p6 cc
    p1 f
    p2 f
    d db Kc7s4h
    p4 cbr 75
    p5 cc
    p6 f
    d db 6s
    p4 cbr 225
    p5 cc
    d db 9d
    p4 cbr 803.50
    p5 cc
    p4 sm AhAc
    p5 sm AsAd

Hand: 3017265406
Players: 6
Winnings: None
Actions:
    d dh p1 ????
    d dh p2 ????
    d dh p3 ????
    d dh p4 ????
    d dh p5 ????
    d dh p6 ????
    p3 cc
    p4 f
    p5 cc
    p6 f
    p1 f
    p2 cc
    d db 3cQhKd
    p2 cbr 30
    p3 cc
    p5 f
    d db Kh
    p2 cbr 60
    p3 cc
    d db 2s
    p2 cc
    p3 cbr 215
    p2 cbr 490
    p3 cc
    p2 sm JdKc
    p3 sm ????

Hand: 3017252372
Players: 2
Winnings: None
Actions:
    d dh p1 ????
    d dh p2 ????
    p2 

NameError: name 'actions' is not defined